In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import resnet50, ResNet50_Weights


# ============================================================
# 1. SETTINGS
# ============================================================

DEVICE = torch.device("cpu")

NUM_KNOWN_CLASSES = 2

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
print("IMPORTED LIBRARIES")

IMPORTED LIBRARIES


In [13]:
# ============================================================
# 2. DATASET
# ============================================================

weights = ResNet50_Weights.DEFAULT

transform = weights.transforms()

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print("DATASET DONE")

DATASET DONE


In [14]:
# ============================================================
# 3. RESNET + TRAINING
# ============================================================

model = resnet50(weights=weights)

model.fc = nn.Linear(
    2048,
    NUM_KNOWN_CLASSES
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

for epoch in range(EPOCHS):

    model.train()

    for input, label in train_loader:

        input = input.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        output = model(input)

        loss = criterion(output, label)

        loss.backward()

        optimizer.step()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"Loss: {loss.item():.4f}"
    )
print("TRAINING DONE")

Epoch 1/10 Loss: 0.5946
Epoch 2/10 Loss: 0.1351
Epoch 3/10 Loss: 0.0288
Epoch 4/10 Loss: 0.0179
Epoch 5/10 Loss: 0.0022
Epoch 6/10 Loss: 0.0004
Epoch 7/10 Loss: 0.0028
Epoch 8/10 Loss: 0.0001
Epoch 9/10 Loss: 0.0001
Epoch 10/10 Loss: 0.0301
TRAINING DONE


In [15]:
# ============================================================
# 4. CONVERT RESNET INTO A FEATURE EXTRACTOR
# ============================================================

# YOU UNDERSTAND THIS
#
# During training:
#
# image → ResNet → 2048 features → FC → 3 class scores
#
# After training, we don't need the FC layer anymore.
#
# nn.Identity() simply returns whatever comes into it.
#
# Therefore:
#
# image → ResNet → 2048 features → Identity → same 2048 features
#
# model(input) now gives [batch_size, 2048]

model.fc = nn.Identity()

model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [16]:
# ============================================================
# 5. EXTRACT FEATURES FROM TRAINING DATA
# ============================================================

# YOU UNDERSTAND MOST OF THIS
#
# DataLoader gives us batches.
#
# For example:
#
# batch 1 → [32, 3, 224, 224]
# batch 2 → [32, 3, 224, 224]
# ...
#
# ResNet gives:
#
# batch 1 → [32, 2048]
# batch 2 → [32, 2048]
#
# We temporarily store those batches in lists.
#
# torch.cat() later joins them together:
#
# [32, 2048]
# [32, 2048]
# [32, 2048]
#      ↓
# [96, 2048]

def get_features(model, loader):

    features_list = []
    labels_list = []

    with torch.no_grad():

        for input, label in loader:
            input = input.to(DEVICE)

            output_features = model(input)

            # YOU HAVE NOT PROPERLY LEARNED NORMALIZATION YET
            #
            # It keeps the same shape.
            #
            # [32, 2048] → [32, 2048]
            #
            # but changes the values so that each feature
            # vector has unit length.

            output_features = nn.functional.normalize(
             output_features,
             p=2,
             dim=1,
             eps=1e-12
             )

            features_list.append(
                output_features
            )

            labels_list.append(
                label
            )

    return (
        torch.cat(features_list),
        torch.cat(labels_list)
    )


train_features, train_labels = get_features(
    model,
    train_loader
)


In [17]:
# ============================================================
# 6. BUILD ONE PROTOTYPE FOR EACH KNOWN CLASS
# ============================================================

# YOU UNDERSTAND THIS SECTION
#
# Suppose:
#
# train_features = [1000, 2048]
# train_labels   = [1000]
#
# We go class by class.
#
# For class 0:
#
# train_labels == 0
#
# creates something like:
#
# [True, True, False, False, True, ...]
#
# Then:
#
# train_features[train_labels == 0]
#
# keeps only the feature vectors belonging to class 0.
#
# If class 0 has 300 images:
#
# [300, 2048]
#
# Then:
#
# mean(dim=0)
#
# averages those 300 feature vectors.
#
# [300, 2048]
#      ↓
# [2048]
#
# That [2048] vector becomes the prototype of class 0.
#
# We repeat this for every class.

prototypes = []

for c in range(NUM_KNOWN_CLASSES):

    class_features = train_features[
        train_labels == c
    ]

    prototype = class_features.mean(
        dim=0
    )

    prototypes.append(
        prototype
    )


# List containing:
#
# prototype 0 → [2048]
# prototype 1 → [2048]
# prototype 2 → [2048]
#
# torch.stack() turns that into:
#
# [3, 2048]

prototypes = torch.stack(
    prototypes
)


# YOU HAVE NOT PROPERLY LEARNED THIS PART YET
#
# We normalize the prototypes as well.

prototypes = nn.functional.normalize(
    prototypes,
    p=2,
    dim=1,
    eps=1e-12
)

In [18]:
# ============================================================
# 7. CALCULATE DISTANCE TO EVERY PROTOTYPE
# ============================================================

# THIS IS WHERE YOU STOPPED UNDERSTANDING
#
# Suppose:
#
# train_features → [1000, 2048]
# prototypes     → [3, 2048]
#
# torch.cdist() calculates the distance between
# every image feature and every prototype.
#
# Result:
#
# distances → [1000, 3]
#
# Meaning:
#
#              class 0    class 1    class 2
# image 0        0.2        0.8        1.1
# image 1        0.3        0.7        1.2
# image 2        0.9        0.2        0.8
# ...
#
# Every row = one image
# Every column = one known class
#
# So each image now has a distance to every known class.

distances = torch.cdist(
    train_features,
    prototypes
)

In [19]:
# ============================================================
# 8. FIND THE CLOSEST KNOWN CLASS
# ============================================================

# NOT PROPERLY LEARNED YET
#
# For every image, we want the smallest distance.
#
# Example:
#
# [0.2, 0.8, 1.1]
#
# minimum = 0.2
#
# Therefore that image is closest to class 0.
#
# dim=1 means:
# look across the class dimension for each image.

min_dists, _ = distances.min(
    dim=1
)


# ============================================================
# 9. FIND THE OSR THRESHOLD
# ============================================================

# NOT LEARNED YET
#
# This is one of the most important OSR concepts.
#
# We need to decide:
#
# "How far is TOO FAR from every known class?"
#
# If an image is close enough → known
#
# If an image is too far from ALL known classes
# → unknown
#
# We use the training distances to determine a threshold.
#
# Here we take the 95th percentile.
#
# This means roughly 95% of training samples have
# a minimum distance below this value.

threshold = torch.quantile(
    min_dists,
    0.95
).item()

print(
    f"OSR Threshold: {threshold:.4f}"
)


OSR Threshold: 0.7405


In [20]:
# ============================================================
# 10. EXTRACT FEATURES FROM TEST DATA
# ============================================================

# YOU UNDERSTAND THE GENERAL FEATURE EXTRACTION IDEA
#
# Same process as training features.
#
# Test images → ResNet → 2048-dimensional features

test_features, test_labels = get_features(
    model,
    test_loader
)


# ============================================================
# 11. COMPARE TEST IMAGES TO KNOWN PROTOTYPES
# ============================================================

# NOT PROPERLY LEARNED YET
#
# Exactly the same idea as before:
#
# test feature → distance to class 0
#              → distance to class 1
#              → distance to class 2

test_distances = torch.cdist(
    test_features,
    prototypes
)


# ============================================================
# 12. FIND CLOSEST CLASS FOR EACH TEST IMAGE
# ============================================================

# NOT PROPERLY LEARNED YET
#
# min_test_dists:
#     smallest distance for each test image
#
# predictions:
#     index of the prototype producing that smallest distance
#
# Example:
#
# distances = [0.2, 0.8, 1.1]
#
# min distance = 0.2
# prototype index = 0
#
# prediction = class 0

min_test_dists, predictions = test_distances.min(
    dim=1
)


# ============================================================
# 13. REJECT IMAGES THAT ARE TOO FAR
# ============================================================

# NOT LEARNED YET
#
# This is the actual UNKNOWN decision.
#
# First the model finds the closest known class.
#
# Then we ask:
#
# "Was it actually close enough?"
#
# If:
#
# min_test_distance <= threshold
#
# → accept the predicted known class
#
# If:
#
# min_test_distance > threshold
#
# → reject it as UNKNOWN
#
# We represent UNKNOWN using -1.

predictions[
    min_test_dists > threshold
] = -1


# ============================================================
# 14. RESULTS
# ============================================================

print("\nRESULTS")
print("--------------------")

print(
    "Predictions:",
    predictions
)

print(
    "Actual:     ",
    test_labels
)


RESULTS
--------------------
Predictions: tensor([ 0,  0,  0,  0,  0,  0,  0, -1, -1, -1,  1,  1,  1,  1, -1,  1,  1])
Actual:      tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2])


In [21]:
print("TRAIN:")
print(train_dataset.classes)
print(train_dataset.class_to_idx)

print("\nTEST:")
print(test_dataset.classes)
print(test_dataset.class_to_idx)

TRAIN:
['red', 'white']
{'red': 0, 'white': 1}

TEST:
['red', 'unknown', 'white']
{'red': 0, 'unknown': 1, 'white': 2}
